In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install datasets

In [ ]:
import pandas as pd
data_file= '/content/drive/MyDrive/TransformesCod/02/ChatBot/estore_qa[1].csv'
df = pd.read_csv(data_file)
df.head()

,question,answer
0,What are the payment options available?,"We accept credit/debit cards, PayPal, and bank..."
1,Do you offer international shipping?,"Yes, we ship to many countries worldwide. Chec..."
2,What is the return policy?,You can return items within 30 days of receipt...
3,How can I track my order?,You can track your order using the tracking nu...
4,What should I do if I receive a damaged item?,Please contact our customer support for assist...


In [ ]:
path_folder = '/content/drive/MyDrive/TransformesCod/02/ChatBot/'

data_file= '/content/drive/MyDrive/TransformesCod/02/ChatBot/estore_qa1.csv'


from datasets import load_dataset
datasets = load_dataset('csv', data_files={'data':data_file},split='data')

print(datasets[:1])


{'question': ['What are the payment options available?'], 'answer': ['We accept credit/debit cards, PayPal, and bank transfers.']}


In [ ]:
from transformers import AutoTokenizer
model_name = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

def tokinizer_function(examples):
  return tokenizer(examples['question'],examples['answer'],truncation=True , padding = 'True' , max_length = 128)

tokenized_datasets = datasets.map(tokinizer_function,remove_columns=['question' , 'answer'])
print(tokenized_datasets[:1])

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

{'input_ids': [[2061, 389, 262, 6074, 3689, 1695, 30, 1135, 2453, 3884, 14, 11275, 270, 4116, 11, 22525, 11, 290, 3331, 16395, 13]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}


In [ ]:
from transformers import AutoModelForCausalLM , TrainingArguments , Trainer , DataCollatorForLanguageModeling
model = AutoModelForCausa1LM.from_pretrained(model_name)
data_collator = DataCollatorForLanguageModeling(tokenizer = tokenizer, mlm=False)
rgs_trainer = TrainingArguments(
  output_dir = path_folder,
  per_device_train_batch_size = 4,
  num_train_epochs = 20,
  learning_rate = 5e-5,
  report_to = 'none',
)

trainer = Trainer(
  model = model,
  args = rgs_trainer,
  train_dataset = tokenized_datasets,
  data_collator = data_collator
)

trainer.train()
model_path = path_folder + 'model'
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
500,0.525528


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/TransformesCod/02/ChatBot/model/tokenizer_config.json',
 '/content/drive/MyDrive/TransformesCod/02/ChatBot/model/tokenizer.json')